# Validate LAL waveform-mode paths

This notebook checks the convention consistency of the LAL mode-generation paths in
`waveformtools.models.lal.LALWaveformModel`.

It is a smoke/diagnostic notebook, not a production accuracy test. It checks that:

1. an available native TD LAL mode approximant returns a `ModesArray` if the local LAL build supports one;
2. `IMRPhenomXPHM` FD modes return a frequency-basis `ModesArray`;
3. the explicit, alias, automatic, and manual FD→TD routes agree;
4. the dimensionless FD→TD route is internally consistent;
5. the TD and FD→TD waveforms can be visually inspected.

Important: `SEOBNRv5PHM` is not necessarily a LALSimulation approximant string in every LALSuite build.
In this codebase it is usually handled through the pySEOBNR/EOB backend, not through `LALWaveformModel`.
Therefore this notebook does **not** use `SEOBNRv5PHM` as the native TD LAL test approximant.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from waveformtools.models.lal import LALWaveformModel
from waveformtools.modes_array import ModesArray

try:
    from lalsimulation import SimInspiralGetApproximantFromString
except Exception:
    SimInspiralGetApproximantFromString = None

## Configuration

In [ ]:
# A clearly FD-domain LAL approximant.
FD_APPROXIMANT = "IMRPhenomXPHM"

# Native TD LAL mode test candidates. The notebook tries them in order and
# skips the native-TD smoke test if none works in the local LALSuite build.
TD_APPROXIMANT_CANDIDATES = [
    "NRSur7dq4",
    "SEOBNRv4PHM",
    "SEOBNRv4HM",
    "IMRPhenomTPHM",
]

BASE_PARAMS = dict(
    mass1=35.0,
    mass2=30.0,
    spin1x=0.0,
    spin1y=0.0,
    spin1z=0.2,
    spin2x=0.0,
    spin2y=0.0,
    spin2z=-0.1,
    distance=400.0,
    inclination=0.7,
    phi_ref=0.3,
    f_lower=20.0,
    f_ref=20.0,
    f_max=512.0,
    delta_t=1.0 / 4096.0,
    delta_f=1.0 / 8.0,
    ell_max=4,
)

MODE_TO_PLOT = (2, 2)

## Helpers

In [ ]:
def make_params(approximant):
    params = dict(BASE_PARAMS)
    params["approximant"] = approximant
    return params


def lal_string_is_available(approximant):
    if SimInspiralGetApproximantFromString is None:
        return False
    try:
        SimInspiralGetApproximantFromString(approximant)
        return True
    except Exception:
        return False


def summarize_modes(wfm, name, max_print=20):
    print(f"\n{name}")
    print("-" * len(name))
    print("type:", type(wfm))
    print("label:", getattr(wfm, "label", None))
    print("ell_max:", getattr(wfm, "ell_max", None))
    print("data_len:", getattr(wfm, "data_len", None))
    try:
        print("time range:", float(wfm.time_axis[0]), float(wfm.time_axis[-1]), "N=", len(wfm.time_axis))
    except Exception as exc:
        print("time range unavailable:", repr(exc))
    try:
        print("freq range:", float(wfm.frequency_axis[0]), float(wfm.frequency_axis[-1]), "N=", len(wfm.frequency_axis))
    except Exception as exc:
        print("freq range unavailable:", repr(exc))

    present = []
    missing = []
    for ell in range(2, int(wfm.ell_max) + 1):
        for m in range(-ell, ell + 1):
            try:
                data = wfm.mode(ell, m)
                if data is None:
                    missing.append((ell, m))
                else:
                    present.append((ell, m))
            except Exception:
                missing.append((ell, m))
    print(f"present modes ({len(present)}):", present[:max_print], "..." if len(present) > max_print else "")
    print(f"missing modes ({len(missing)}):", missing[:max_print], "..." if len(missing) > max_print else "")
    return present, missing


def relative_l2(a, b, eps=1e-300):
    a = np.asarray(a)
    b = np.asarray(b)
    return np.linalg.norm(a - b) / max(np.linalg.norm(a), np.linalg.norm(b), eps)


def compare_common_modes(a, b, label_a="a", label_b="b", rtol=1e-10, atol=1e-12, max_print=20):
    common = []
    rows = []
    ell_max = min(int(a.ell_max), int(b.ell_max))
    for ell in range(2, ell_max + 1):
        for m in range(-ell, ell + 1):
            try:
                am = np.asarray(a.mode(ell, m))
                bm = np.asarray(b.mode(ell, m))
            except Exception:
                continue
            if am.shape != bm.shape:
                rows.append((ell, m, "shape_mismatch", am.shape, bm.shape, np.nan, np.nan))
                continue
            err_abs = np.max(np.abs(am - bm))
            err_rel = relative_l2(am, bm)
            ok = np.allclose(am, bm, rtol=rtol, atol=atol)
            rows.append((ell, m, ok, am.shape, bm.shape, float(err_abs), float(err_rel)))
            common.append((ell, m))
    failures = [r for r in rows if r[2] is not True]
    print(f"Compared {len(common)} common modes: {label_a} vs {label_b}")
    print(f"Failures / mismatches: {len(failures)}")
    for row in failures[:max_print]:
        print(row)
    return rows, failures

In [ ]:
def peak_centered_time(wfm, mode=MODE_TO_PLOT):
    y = np.asarray(wfm.mode(*mode))
    t = np.asarray(wfm.time_axis)
    if len(t) != len(y):
        return np.arange(len(y))
    peak = int(np.argmax(np.abs(y)))
    return t - t[peak]


def normalized_mode(wfm, mode=MODE_TO_PLOT):
    y = np.asarray(wfm.mode(*mode))
    scale = np.max(np.abs(y))
    if scale == 0 or not np.isfinite(scale):
        scale = 1.0
    return y / scale


def plot_mode_real(waveforms, title, mode=MODE_TO_PLOT, normalize=False, peak_center=True, xlim=None):
    plt.figure(figsize=(8, 3.5))
    for label, wfm in waveforms:
        if wfm is None:
            continue
        y = normalized_mode(wfm, mode=mode) if normalize else np.asarray(wfm.mode(*mode))
        x = peak_centered_time(wfm, mode=mode) if peak_center else np.asarray(wfm.time_axis)
        plt.plot(x, y.real, label=label)
    if xlim is not None:
        plt.xlim(*xlim)
    plt.xlabel("t - t_peak" if peak_center else "time")
    plt.ylabel(f"Re h_{{{mode[0]}{mode[1]}}}" + (" / max|h|" if normalize else ""))
    plt.title(title)
    plt.legend()
    plt.tight_layout()


def plot_mode_amp(waveforms, title, mode=MODE_TO_PLOT, normalize=True, peak_center=True, xlim=None):
    plt.figure(figsize=(8, 3.5))
    for label, wfm in waveforms:
        if wfm is None:
            continue
        y = normalized_mode(wfm, mode=mode) if normalize else np.asarray(wfm.mode(*mode))
        x = peak_centered_time(wfm, mode=mode) if peak_center else np.asarray(wfm.time_axis)
        plt.plot(x, np.abs(y), label=label)
    if xlim is not None:
        plt.xlim(*xlim)
    plt.xlabel("t - t_peak" if peak_center else "time")
    plt.ylabel(f"|h_{{{mode[0]}{mode[1]}}}|" + (" / max|h|" if normalize else ""))
    plt.title(title)
    plt.legend()
    plt.tight_layout()


def plot_route_difference(reference, other, ref_label, other_label, mode=MODE_TO_PLOT):
    ref = np.asarray(reference.mode(*mode))
    oth = np.asarray(other.mode(*mode))
    n = min(len(ref), len(oth))
    diff = oth[:n] - ref[:n]
    t = np.asarray(reference.time_axis[:n])
    plt.figure(figsize=(8, 3.2))
    plt.plot(t, diff.real, label="Re difference")
    plt.plot(t, diff.imag, label="Im difference")
    plt.xlabel("time")
    plt.ylabel(f"{other_label} - {ref_label}")
    plt.title(f"FD→TD route difference for h_{{{mode[0]}{mode[1]}}}")
    plt.legend()
    plt.tight_layout()

## 1. Optional native TD LAL-mode path smoke test

This is deliberately optional because available LAL mode approximants depend on the local LALSuite build and data files.

In [ ]:
w_td_native = None
td_model = None
TD_APPROXIMANT_USED = None
td_attempt_errors = {}

for candidate in TD_APPROXIMANT_CANDIDATES:
    if not lal_string_is_available(candidate):
        td_attempt_errors[candidate] = "not accepted by SimInspiralGetApproximantFromString"
        continue
    try:
        td_model = LALWaveformModel(parameters_dict=make_params(candidate))
        w_td_native = td_model.get_td_waveform_modes(dimensionless=False)
        TD_APPROXIMANT_USED = candidate
        break
    except Exception as exc:
        td_attempt_errors[candidate] = repr(exc)

if w_td_native is None:
    print("SKIP native TD LAL-mode test. No candidate worked in this environment.")
    print(td_attempt_errors)
else:
    assert isinstance(w_td_native, ModesArray)
    present_td, missing_td = summarize_modes(w_td_native, f"native TD modes: {TD_APPROXIMANT_USED}")

## 2. FD modes and FD modes as TD return `ModesArray`

In [ ]:
assert lal_string_is_available(FD_APPROXIMANT), f"{FD_APPROXIMANT} is not available in this LALSuite build"

fd_params = make_params(FD_APPROXIMANT)
fd_model = LALWaveformModel(parameters_dict=fd_params)

w_fd = fd_model.get_fd_waveform_modes(dimensionless=False)
w_fd_as_td = fd_model.get_fd_waveform_modes_as_td(dimensionless=False)
w_fd_as_td_alias = fd_model.get_fd_modes_as_td(dimensionless=False)
w_fd_auto_td = fd_model.get_td_waveform_modes(dimensionless=False)
w_fd_manual_as_td = w_fd.to_time_basis()

assert isinstance(w_fd, ModesArray)
assert isinstance(w_fd_as_td, ModesArray)
assert isinstance(w_fd_as_td_alias, ModesArray)
assert isinstance(w_fd_auto_td, ModesArray)
assert isinstance(w_fd_manual_as_td, ModesArray)

present_fd, missing_fd = summarize_modes(w_fd, f"native FD modes: {FD_APPROXIMANT}")
present_fd_td, missing_fd_td = summarize_modes(w_fd_as_td, f"FD modes as TD: {FD_APPROXIMANT}")
present_fd_manual_td, missing_fd_manual_td = summarize_modes(w_fd_manual_as_td, f"manual FD.to_time_basis(): {FD_APPROXIMANT}")

## 3. Compare the FD→TD implementations/routes

The routes compared here are:

- explicit `get_fd_waveform_modes_as_td`;
- alias `get_fd_modes_as_td`;
- automatic `get_td_waveform_modes` for an FD approximant;
- manual `get_fd_waveform_modes(...).to_time_basis()`.

In [ ]:
comparisons = [
    (w_fd_as_td, w_fd_as_td_alias, "explicit FD→TD", "alias FD→TD"),
    (w_fd_as_td, w_fd_auto_td, "explicit FD→TD", "automatic get_td_waveform_modes"),
    (w_fd_as_td, w_fd_manual_as_td, "explicit FD→TD", "manual FD.to_time_basis"),
]

all_failures = {}
for a, b, la, lb in comparisons:
    rows, failures = compare_common_modes(a, b, label_a=la, label_b=lb, rtol=0.0, atol=0.0)
    all_failures[(la, lb)] = failures

# The first two should be exactly identical if the public routes delegate correctly.
assert len(all_failures[("explicit FD→TD", "alias FD→TD")]) == 0
assert len(all_failures[("explicit FD→TD", "automatic get_td_waveform_modes")]) == 0

# The manual route should also normally match. If this fails, inspect whether one route
# applies extra metadata/normalization or axis handling beyond to_time_basis().
assert len(all_failures[("explicit FD→TD", "manual FD.to_time_basis")]) == 0

## 4. Dimensionless convention check

In [ ]:
w_fd_auto_td_dimless = fd_model.get_td_waveform_modes(dimensionless=True)
w_fd_as_td_dimless = fd_model.get_fd_waveform_modes_as_td(dimensionless=True)

rows, failures = compare_common_modes(
    w_fd_auto_td_dimless,
    w_fd_as_td_dimless,
    label_a="dimensionless get_td_waveform_modes on FD approximant",
    label_b="dimensionless get_fd_waveform_modes_as_td",
    rtol=0.0,
    atol=0.0,
)
assert len(failures) == 0

## 5. Visual check: native TD approximant

This cell plots the selected native TD LAL approximant if one was available.

In [ ]:
if w_td_native is None:
    print("No native TD LAL-mode waveform available to plot in this environment.")
else:
    plot_mode_real(
        [(f"native TD: {TD_APPROXIMANT_USED}", w_td_native)],
        title=f"Native TD LAL mode waveform: {TD_APPROXIMANT_USED}",
        normalize=False,
        peak_center=True,
    )
    plot_mode_amp(
        [(f"native TD: {TD_APPROXIMANT_USED}", w_td_native)],
        title=f"Native TD LAL mode amplitude: {TD_APPROXIMANT_USED}",
        normalize=True,
        peak_center=True,
    )

## 6. Visual check: FD→TD implementations

These curves should visually lie on top of each other if the FD→TD routes are consistent.

In [ ]:
fd_td_routes = [
    ("explicit get_fd_waveform_modes_as_td", w_fd_as_td),
    ("alias get_fd_modes_as_td", w_fd_as_td_alias),
    ("automatic get_td_waveform_modes", w_fd_auto_td),
    ("manual FD.to_time_basis", w_fd_manual_as_td),
]

plot_mode_real(
    fd_td_routes,
    title=f"FD→TD route comparison: {FD_APPROXIMANT}",
    normalize=False,
    peak_center=True,
)
plot_mode_amp(
    fd_td_routes,
    title=f"FD→TD route amplitude comparison: {FD_APPROXIMANT}",
    normalize=True,
    peak_center=True,
)

## 7. Visual check: TD approximant vs FD→TD approximant

This is a qualitative comparison only because the two curves may come from different approximants.
The amplitude-normalized, peak-centered plot is useful for checking gross time-domain conventions.

In [ ]:
if w_td_native is None:
    print("No native TD LAL-mode waveform available; skipping TD-vs-FD→TD visual overlay.")
else:
    plot_mode_real(
        [
            (f"native TD: {TD_APPROXIMANT_USED}", w_td_native),
            (f"FD→TD: {FD_APPROXIMANT}", w_fd_as_td),
        ],
        title="Qualitative TD approximant vs FD→TD approximant comparison",
        normalize=True,
        peak_center=True,
        xlim=(-500, 100),
    )
    plot_mode_amp(
        [
            (f"native TD: {TD_APPROXIMANT_USED}", w_td_native),
            (f"FD→TD: {FD_APPROXIMANT}", w_fd_as_td),
        ],
        title="Qualitative TD approximant vs FD→TD approximant amplitude comparison",
        normalize=True,
        peak_center=True,
        xlim=(-500, 100),
    )

## 8. Difference plots between FD→TD routes

These should be zero or at numerical roundoff if the implementations are equivalent.

In [ ]:
plot_route_difference(w_fd_as_td, w_fd_as_td_alias, "explicit", "alias")
plot_route_difference(w_fd_as_td, w_fd_auto_td, "explicit", "automatic")
plot_route_difference(w_fd_as_td, w_fd_manual_as_td, "explicit", "manual")